# Day-Ahead PJM Load Forecasting with a Time-Series Foundation Model

This notebook forecasts hourly electricity demand in the PJM East region 24 hours ahead with **Chronos-Bolt**, a pretrained
time-series foundation model, and compares three ways of using it:

1. **Zero-shot**: the pretrained model is applied directly, with no training on PJM data.
2. **Fine-tuned**: the pretrained weights are further trained on the PJM training period.
3. **Baselines**: a seasonal naive forecast (same hour yesterday) and a gradient-boosted tree model (LightGBM) on lagged features.

All models are evaluated with a **rolling-origin backtest**: every day of a held-out 14-day test period is forecast using only
the data available before that day, so there is no look-ahead. Accuracy is reported as MAPE and RMSE.

**Runtime:** about 15 to 20 minutes on a free Colab GPU (Runtime > Change runtime type > T4 GPU), including installation.

In [ ]:
# Install AutoGluon TimeSeries (includes Chronos). In Colab, if you are asked to restart the session
# after installation, restart it and continue from the next cell.
%pip install -q autogluon.timeseries

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams['figure.figsize'] = (11, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('GPU available:', torch.cuda.is_available())

## 1. Data

Hourly PJM East (PJME) demand from January to May 2017, the same real demand series used in the
[battery arbitrage project](https://github.com/ekahorsu/battery-arbitrage-reinforcement-learning).
The file ships with this repository; if it is missing (for example in a fresh Colab session) it is downloaded from GitHub.

In [ ]:
LOCAL = 'pjm_east_hourly_load.csv'
REMOTE = 'https://raw.githubusercontent.com/ekahorsu/battery-arbitrage-reinforcement-learning/main/power_prices.csv'

if os.path.exists(LOCAL):
    raw = pd.read_csv(LOCAL, parse_dates=['Datetime'])
else:
    raw = pd.read_csv(REMOTE, parse_dates=['Datetime'])[['Datetime', 'PJME_MW']].rename(columns={'PJME_MW': 'load_MW'})

# The series has one missing hour at the daylight saving time change. Put it on a regular hourly grid
# and fill the single gap by time interpolation.
load = raw.set_index('Datetime')['load_MW'].sort_index()
load = load[~load.index.duplicated()].asfreq('h').interpolate(method='time')
load = load.iloc[: len(load) // 24 * 24]   # keep whole days so every test forecast starts at midnight

print(f'Observations: {len(load)} hours, {load.index[0]} to {load.index[-1]}')
print(f'Load range: {load.min():,.0f} to {load.max():,.0f} MW, mean {load.mean():,.0f} MW')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
load.plot(ax=axes[0], linewidth=0.6)
axes[0].set_title('PJM East hourly load')
axes[0].set_ylabel('MW')
load.iloc[-168:].plot(ax=axes[1], linewidth=1.2)
axes[1].set_title('Last week: daily and weekly cycles')
axes[1].set_ylabel('MW')
plt.tight_layout()
plt.show()

## 2. Train and test split

The last 14 days (336 hours) are held out for testing. Every model is trained only on the earlier period.

In [ ]:
H = 24                 # forecast horizon: the next 24 hours
TEST_DAYS = 14
test_start = len(load) - TEST_DAYS * H

def to_tsdf(series):
    df = pd.DataFrame({'item_id': 'PJME', 'timestamp': series.index, 'target': series.values})
    return TimeSeriesDataFrame.from_data_frame(df, id_column='item_id', timestamp_column='timestamp')

train = to_tsdf(load.iloc[:test_start])
print(f'Train: {test_start} hours ({load.index[0].date()} to {load.index[test_start - 1].date()})')
print(f'Test:  {TEST_DAYS * H} hours ({load.index[test_start].date()} to {load.index[-1].date()})')

## 3. Models

AutoGluon trains all four models with the same interface. Chronos-Bolt (small, about 48 million parameters) is loaded
from its pretrained checkpoint twice: once used as is (zero-shot) and once fine-tuned on the training period.

In [ ]:
t0 = time.time()
predictor = TimeSeriesPredictor(
    prediction_length=H,
    target='target',
    eval_metric='MAPE',
    path='ag_pjm_models',
    verbosity=2,
).fit(
    train,
    hyperparameters={
        'SeasonalNaive': {},
        'DirectTabular': {},
        'Chronos': [
            {'model_path': 'bolt_small', 'ag_args': {'name_suffix': 'ZeroShot'}},
            {'model_path': 'bolt_small', 'fine_tune': True, 'ag_args': {'name_suffix': 'FineTuned'}},
        ],
    },
    enable_ensemble=False,
    random_seed=SEED,
)
print(f'Training time: {time.time() - t0:.0f} s')
MODELS = predictor.model_names()
print(MODELS)

## 4. Rolling-origin backtest

For each of the 14 test days, every model receives all load data up to midnight and forecasts the next 24 hours.
The forecasts are then compared with the actual load.

In [ ]:
def mape(y, f):
    return 100 * np.mean(np.abs((y - f) / y))

def rmse(y, f):
    return np.sqrt(np.mean((y - f) ** 2))

records, forecasts = [], {m: [] for m in MODELS}
for d in range(TEST_DAYS):
    cutoff = test_start + d * H
    context = to_tsdf(load.iloc[:cutoff])
    actual = load.iloc[cutoff:cutoff + H].values
    for m in MODELS:
        pred = predictor.predict(context, model=m)
        f = pred['mean'].values
        forecasts[m].append(pred.reset_index(drop=True).assign(timestamp=load.index[cutoff:cutoff + H]))
        records.append({'day': load.index[cutoff].date(), 'model': m, 'MAPE (%)': mape(actual, f), 'RMSE (MW)': rmse(actual, f)})

daily = pd.DataFrame(records)
results = (daily.groupby('model')[['MAPE (%)', 'RMSE (MW)']].mean()
           .sort_values('MAPE (%)').round({'MAPE (%)': 2, 'RMSE (MW)': 0}))
results

In [ ]:
def short(m):
    for key, name in [('FineTuned', 'Chronos-Bolt fine-tuned'), ('ZeroShot', 'Chronos-Bolt zero-shot'),
                      ('SeasonalNaive', 'Seasonal naive'), ('DirectTabular', 'LightGBM (DirectTabular)')]:
        if key in m:
            return name
    return m

summary = results.rename(index=short)
naive = summary.loc['Seasonal naive', 'MAPE (%)']
summary['MAPE reduction vs seasonal naive (%)'] = (100 * (naive - summary['MAPE (%)']) / naive).round(1)
print(summary.to_string())

ft, zs = summary.loc['Chronos-Bolt fine-tuned', 'MAPE (%)'], summary.loc['Chronos-Bolt zero-shot', 'MAPE (%)']
print(f'\nFine-tuning changed MAPE from {zs:.2f}% (zero-shot) to {ft:.2f}%, '
      f'a {100 * (zs - ft) / zs:.1f}% relative reduction.')

In [ ]:
# Daily MAPE across the test period
pivot = daily.assign(model=daily['model'].map(short)).pivot(index='day', columns='model', values='MAPE (%)')
pivot.plot(marker='o')
plt.ylabel('MAPE (%)')
plt.title('Daily forecast error over the 14-day test period')
plt.tight_layout()
plt.show()

In [ ]:
# Last three test days: actual load against the Chronos forecasts, with the 10-90% band of the fine-tuned model
fine = next(m for m in MODELS if 'FineTuned' in m)
zero = next(m for m in MODELS if 'ZeroShot' in m)
ft_df = pd.concat(forecasts[fine][-3:]).set_index('timestamp')
zs_df = pd.concat(forecasts[zero][-3:]).set_index('timestamp')

plt.plot(load.loc[ft_df.index], color='black', linewidth=2, label='Actual load')
plt.plot(zs_df['mean'], '--', label='Chronos-Bolt zero-shot')
plt.plot(ft_df['mean'], label='Chronos-Bolt fine-tuned')
plt.fill_between(ft_df.index, ft_df['0.1'], ft_df['0.9'], alpha=0.2, label='Fine-tuned 10-90% interval')
plt.ylabel('MW')
plt.title('Day-ahead forecasts over the last three test days')
plt.legend()
plt.tight_layout()
plt.show()

inside = np.mean([(load.loc[f.set_index('timestamp').index].values >= f['0.1'].values) &
                  (load.loc[f.set_index('timestamp').index].values <= f['0.9'].values) for f in forecasts[fine]])
print(f'Share of test hours inside the fine-tuned 10-90% interval: {100 * inside:.0f}% (ideal: 80%)')

## 5. Takeaways

- The rolling-origin backtest gives an honest estimate of day-ahead accuracy: each forecast uses only data available before the forecast day.
- Comparing zero-shot and fine-tuned Chronos-Bolt shows how much a pretrained foundation model gains from a few minutes of domain-specific training on PJM load.
- The seasonal naive forecast is a strong reference for load, because demand repeats its daily shape; a useful model has to beat it clearly.
- Chronos also produces probabilistic forecasts, and the 10-90% interval coverage checks whether that uncertainty is well calibrated.

**Next steps:** add weather (temperature) as a covariate, extend the data to several years, and fine-tune larger Chronos models on a GPU cluster.